# L1 — Data & Knowledge · Unstructured Data

## What you'll learn

- How the agents' data splits into structured orders and unstructured policy text
- How to load the return-policy documents into a vector store of your choice: Amazon Bedrock Knowledge Bases or Couchbase
- How to retrieve the passages that best match a question, to confirm the store works

## Overview

The AnyCompany agents use two kinds of data:

- **Structured data: customer orders.** Order IDs, products, prices, dates, delivery status and membership tier. These fixed-field records are in an **Amazon DynamoDB** table that the workshop creates and fills for you, so there is nothing to do. The **Order Agent** looks orders up there.
- **Unstructured data: the return policy.** Free-form text that customers ask about in their own words, so it is searched by meaning rather than keywords. This notebook prepares it for the **Refund Agent**.

Preparing the policy takes three steps:

1. **Embed:** split the policy into passages and turn each into a vector with Amazon Titan.
2. **Store:** save the vectors in a vector store.
3. **Retrieve:** embed a question the same way and return the closest passages.

The Refund Agent answers from the retrieved passages, which is retrieval-augmented generation (RAG). You pick the vector store in the **Partner Selection UI** (Data → Vector Store) and paste its code into the two empty cells below.

## Architecture

![Data and knowledge layer: the same policy documents go either to a Bedrock Knowledge Base backed by OpenSearch Serverless or, as the partner option, are chunked and embedded in the notebook and stored in Couchbase Capella, both using Titan Embeddings V2. The Refund tool retrieves from whichever store was chosen for the Refund Agent. The pre-provisioned DynamoDB CustomerOrders table is queried by the Order tool for the Order Agent.](images/l1-data-knowledge-architecture.png)

This notebook builds the two paths at the top: a Bedrock Knowledge Base or Couchbase. The DynamoDB orders table at the bottom is already in place, and the tools that read both stores are added in L3.

## Steps

0. Install the packages
1. Load the return-policy documents
2. Create the vector store and add the documents (code from the Partner Selection UI)
3. Ask a test question and check the passages that come back (code from the Partner Selection UI)

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

The Partner Selection UI from L0 ([`1_partner_selection.ipynb`](../l0-partner-selection/1_partner_selection.ipynb)) is running.

## Step 0: Install the packages

Install the packages for both vector stores. Run this once, at the start.

In [ ]:
!pip install -q --upgrade boto3 opensearch-py==2.7.0 requests-aws4auth==1.3.0 couchbase==4.3.5 requests==2.32.4

## Step 1: Load the return-policy documents

Set the AWS Region and read the return-policy documents from `sample-data/return-policy/`. This step is the same whichever vector store you choose, and both start from the same documents. Anything specific to one store, such as uploading the documents to Amazon S3 for Bedrock or connecting to the Couchbase cluster, is in the code you paste in Step 2.

In [ ]:
import os, glob
import boto3

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
ssm = boto3.client("ssm", region_name=REGION)
print(f"Region: {REGION}")

# Load the return-policy documents from local sample data (shared across providers).
POLICY_DOCS = {}
for path in glob.glob("./sample-data/return-policy/*"):
    with open(path) as f:
        POLICY_DOCS[os.path.basename(path)] = f.read()
print(f"\nLoaded {len(POLICY_DOCS)} policy document(s): {list(POLICY_DOCS)}")

print("\nStep 1 complete — POLICY_DOCS loaded, region + SSM client ready.")


## Step 2: Create the vector store and add the documents

Paste the **1 · Ingest the knowledge** code from the Partner Selection UI (Data → Vector Store), then run it. It creates the vector store, embeds and indexes the policy documents, and saves the store's details for L3.


<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Data → Vector Store</b>.</li>
<li>Pick your provider card, and save its API key if it asks for one.</li><li>Expand <b>📋 1 · Ingest the knowledge</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "1 · Ingest the knowledge" snippet from the Partner Selection UI here, then run.



## Step 3: Ask a test question

Paste the **2 · Test retrieval** code from the Partner Selection UI, then run it. It asks a sample policy question and shows the passages that come back, which confirms the store is ready for the agents.

<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Data → Vector Store</b>.</li>
<li>Expand <b>📋 2 · Test retrieval</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "2 · Test retrieval" snippet from the Partner Selection UI here, then run.



## Summary

This notebook loaded the return-policy documents into your vector store (Amazon Bedrock Knowledge Bases or Couchbase) and retrieved passages for a sample question. The customer orders were already in DynamoDB, so the data layer now gives the agents both order lookups and policy search. The store details are saved for L3.

**Next:** [`1_pluggable_inference_layer.ipynb`](../l2-inference/1_pluggable_inference_layer.ipynb). Choose the model for each agent role.

---
## Cleanup

Deletes what this notebook created. For Bedrock Knowledge Bases, that is the Knowledge Base, its OpenSearch Serverless collection, policies and VPC endpoint, the policy documents in the workshop bucket, and the saved IDs. For Couchbase, it deletes the saved connection details; delete the Capella cluster yourself in the Capella console. The bucket and the IAM role belong to the workshop stack and are removed when it's deleted.

Run this **last**, after the cleanup in [`1_gateway_tools.ipynb`](../l3-orchestration/1_gateway_tools.ipynb), because the Refund tool reads from this store. Uncomment the cell to run it.

In [ ]:
## === CLEANUP: delete the vector store created in this notebook ===
## Uncomment and run ONLY after you are done with ALL notebooks, and AFTER the
## 1_gateway_tools.ipynb cleanup (the Refund tool reads from this store).
# import boto3, os, time
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# ssm = boto3.client("ssm", region_name=REGION)
#
# def _ssm_get(key):
#     try: return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound: return None
#
# def _ssm_delete(keys):
#     for key in keys:
#         try: ssm.delete_parameter(Name=f"{SSM_PREFIX}/{key}"); print(f"Deleted SSM: {key}")
#         except ssm.exceptions.ParameterNotFound: pass
#
# PROVIDER = _ssm_get("kb_provider") or "bedrock"
# print(f"Vector store: {PROVIDER}")
#
# if PROVIDER == "bedrock":
#     COLLECTION = "composable-agents-collection"
#     agent = boto3.client("bedrock-agent", region_name=REGION)
#     aoss = boto3.client("opensearchserverless", region_name=REGION)
#     s3 = boto3.client("s3", region_name=REGION)
#
#     # 1. Knowledge Base and its data source
#     kb_ids = [k["knowledgeBaseId"] for p in agent.get_paginator("list_knowledge_bases").paginate()
#               for k in p["knowledgeBaseSummaries"] if k["name"] == "composable-agents-kb"]
#     for kb_id in kb_ids:
#         for ds in agent.list_data_sources(knowledgeBaseId=kb_id)["dataSourceSummaries"]:
#             agent.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds["dataSourceId"])
#         agent.delete_knowledge_base(knowledgeBaseId=kb_id)
#         for _ in range(60):
#             try: agent.get_knowledge_base(knowledgeBaseId=kb_id); time.sleep(5)
#             except agent.exceptions.ResourceNotFoundException: break
#         print(f"Deleted Knowledge Base: {kb_id}")
#
#     # 2. OpenSearch Serverless collection, its policies and the VPC endpoint
#     for c in aoss.list_collections(collectionFilters={"name": COLLECTION})["collectionSummaries"]:
#         aoss.delete_collection(id=c["id"]); print(f"Deleted collection: {COLLECTION}")
#     for name, kind in [(f"{COLLECTION}-enc", "encryption"), (f"{COLLECTION}-net", "network")]:
#         try: aoss.delete_security_policy(name=name, type=kind); print(f"Deleted {kind} policy: {name}")
#         except aoss.exceptions.ResourceNotFoundException: pass
#     try:
#         aoss.delete_access_policy(name=f"{COLLECTION[:16]}-access", type="data")
#         print(f"Deleted data access policy: {COLLECTION[:16]}-access")
#     except aoss.exceptions.ResourceNotFoundException: pass
#     for v in aoss.list_vpc_endpoints(vpcEndpointFilters={"status": "ACTIVE"})["vpcEndpointSummaries"]:
#         if v["name"] == "workshop-aoss-vpce":
#             aoss.delete_vpc_endpoint(id=v["id"]); print("Deleted VPC endpoint: workshop-aoss-vpce")
#
#     # 3. The policy documents uploaded to the workshop bucket (the bucket belongs to the stack)
#     bucket = _ssm_get("s3_bucket")
#     if bucket:
#         objs = s3.list_objects_v2(Bucket=bucket, Prefix="return-policy/").get("Contents", [])
#         if objs:
#             s3.delete_objects(Bucket=bucket, Delete={"Objects": [{"Key": o["Key"]} for o in objs]})
#             print(f"Deleted {len(objs)} policy documents from s3://{bucket}/return-policy/")
#
#     _ssm_delete(["kb_id", "kb_provider"])
#
# else:
#     # Couchbase: the Capella cluster is in your Capella account. Delete it in the
#     # Capella console (cloud.couchbase.com). This removes the saved connection details.
#     _ssm_delete(["kb_provider", "kb_chunks_keyspace", "kb_vector_index", "kb_embedding_model",
#                  "kb_embedding_dimensions", "kb_similarity"])
#     _ssm_delete([f"couchbase/{k}" for k in ["connection_string", "username", "password", "bucket",
#                  "support_scope", "orders_collection", "knowledge_scope", "chunks_collection",
#                  "cluster_id"]])
#
# print("\nL1 vector store cleaned up.")